# AgenticEd Labs · Week 1 — Agent Fundamentals: From API Call to Agent Loop

<a href="https://colab.research.google.com/github/AgenticEdOrg/AgenticEd/blob/main/labs/week1_agent_fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

| | |
|---|---|
| **Time** | 75–90 minutes |
| **Approx. API cost** | ≈ US$0.05 with the default low-cost model |
| **Prerequisites** | Python; `00_environment_check` passing |
| **You'll use** | LiteLLM, Pydantic AI, OpenAI Agents SDK, Open-Meteo API, Wikipedia API |

**Learning objectives**
- Make and meter a raw model call (tokens, latency, cost)
- Write tool schemas by hand and implement the **tool-calling loop** from scratch
- Measure the cost of an **agent** against a fixed **workflow** for the same task
- Rebuild the agent in **Pydantic AI** (typed tools, structured output, usage limits, approval-gated tools)
- Compare with the **OpenAI Agents SDK** and understand what frameworks give you

> **Before you start:** add `OPENAI_API_KEY` *or* `ANTHROPIC_API_KEY` to Colab Secrets (key icon in the left sidebar → enable notebook access), or to `labs/.env` locally. See `00_environment_check.ipynb` if anything fails.

In [ ]:
# ── Setup (run once) ─────────────────────────────────────────────────────────
# In Colab: clones the course repo and installs the pinned requirements (2–3 minutes the first time).
# If Colab asks you to restart the runtime after installing, do so and run this cell again.
import sys, pathlib
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("/content/AgenticEd").exists():
        !git clone -q --depth 1 https://github.com/AgenticEdOrg/AgenticEd.git /content/AgenticEd
    %cd /content/AgenticEd/labs
    %pip install -q -r requirements.txt
sys.path.insert(0, str(pathlib.Path.cwd()))

import labkit
cfg = labkit.setup()          # reads your key, picks the model, prints what it chose

## 1 · One model call, metered

Everything an agent does is built from calls like this. Track three numbers from day one: **tokens**, **latency**, and **cost**. LiteLLM gives every provider the same OpenAI-style interface.

In [ ]:
import json, time, litellm
litellm.drop_params = True            # silently drop settings a given model doesn't support
litellm.suppress_debug_info = True

t = time.time()
resp = litellm.completion(
    model=cfg.litellm,
    messages=[{"role": "user", "content": "In one sentence: what makes software an 'agent' rather than a tool?"}],
    **cfg.litellm_kwargs())
u = resp.usage
print(resp.choices[0].message.content)
print(f"\n{u.prompt_tokens} in / {u.completion_tokens} out tokens · {time.time()-t:.1f}s · "
      f"~${cfg.cost(u.prompt_tokens, u.completion_tokens) or 'n/a'}")

## 2 · Real tools

Tools are ordinary functions that reach the outside world. These two call free public APIs (no key): **Open-Meteo** for weather and **Wikipedia** for facts. Note the defensive habits: timeouts, a User-Agent, and errors returned as data instead of raised — the model can read an error and recover; a crash ends the run.

In [ ]:
import requests
UA = {"User-Agent": "AgenticEd-labs/1.0 (education; https://agenticedacademy.com)"}

def get_forecast(city: str) -> dict:
    """3-day forecast (max/min °C and max rain probability) for a city."""
    try:
        geo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                           params={"name": city, "count": 1}, headers=UA, timeout=15).json()
        if not geo.get("results"):
            return {"error": f"city not found: {city}"}
        p = geo["results"][0]
        d = requests.get("https://api.open-meteo.com/v1/forecast", headers=UA, timeout=15, params={
            "latitude": p["latitude"], "longitude": p["longitude"], "timezone": "auto", "forecast_days": 3,
            "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max"}).json()["daily"]
        return {"city": p["name"], "country": p.get("country"), "days": [
            {"date": d["time"][i], "max_c": d["temperature_2m_max"][i], "min_c": d["temperature_2m_min"][i],
             "rain_chance_pct": d["precipitation_probability_max"][i]} for i in range(len(d["time"]))]}
    except requests.RequestException as e:
        return {"error": f"weather service unavailable: {e}"}

def wikipedia_summary(title: str) -> dict:
    """First paragraph of a Wikipedia article."""
    try:
        r = requests.get("https://en.wikipedia.org/api/rest_v1/page/summary/" + requests.utils.quote(title.replace(" ", "_")),
                         headers=UA, timeout=15)
        if r.status_code == 404:
            return {"error": f"no article titled {title!r}"}
        return {"title": r.json().get("title"), "extract": r.json().get("extract", "")[:1200]}
    except requests.RequestException as e:
        return {"error": f"wikipedia unavailable: {e}"}

print(json.dumps(get_forecast("Chennai"), indent=1)[:400])
print(wikipedia_summary("Chennai")["extract"][:200], "…")

## 3 · Tool schemas by hand

The model never runs your code. It sees a **JSON Schema** for each tool, and replies with a structured request to call one. Writing a schema by hand once shows you exactly what frameworks generate for you later.

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "get_forecast",
        "description": "Get a 3-day weather forecast for a city. Use for any question about weather or packing.",
        "parameters": {"type": "object", "properties": {"city": {"type": "string", "description": "City name, e.g. 'Oslo'"}},
                       "required": ["city"]}}},
    {"type": "function", "function": {
        "name": "wikipedia_summary",
        "description": "Get the opening paragraph of a Wikipedia article. Use for facts about places, people, and events.",
        "parameters": {"type": "object", "properties": {"title": {"type": "string", "description": "Exact article title"}},
                       "required": ["title"]}}},
]
REGISTRY = {"get_forecast": get_forecast, "wikipedia_summary": wikipedia_summary}

resp = litellm.completion(model=cfg.litellm, tools=TOOLS, **cfg.litellm_kwargs(),
                          messages=[{"role": "user", "content": "Will it rain in Oslo in the next three days?"}])
msg = resp.choices[0].message
print("The model replied with a tool request, not an answer:")
for c in msg.tool_calls or []:
    print(f"  id={c.id}  name={c.function.name}  arguments={c.function.arguments}")

## 4 · The agent loop, from scratch

This is the core of every agent framework: **call the model → if it requests tools, run them and append the results → repeat until it answers or a limit is hit.** Read it closely; everything later in the course is an elaboration of these ~40 lines.

In [ ]:
SYSTEM = ("You are a careful trip-planning assistant for a school. Use tools for any weather or factual claim; "
          "never guess. Keep answers short and practical.")

def run_agent(task, tools=TOOLS, registry=REGISTRY, system=SYSTEM, max_steps=6, verbose=True):
    """Tool-calling loop. Returns (answer, trace) where trace lists every model and tool call."""
    messages = [{"role": "system", "content": system}, {"role": "user", "content": task}]
    trace = []
    for step in range(1, max_steps + 1):
        t = time.time()
        resp = litellm.completion(model=cfg.litellm, messages=messages, tools=tools, **cfg.litellm_kwargs())
        msg, u = resp.choices[0].message, resp.usage
        trace.append({"step": step, "kind": "llm", "in_tokens": u.prompt_tokens, "out_tokens": u.completion_tokens,
                      "seconds": round(time.time() - t, 2)})
        assistant = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant["tool_calls"] = [{"id": c.id, "type": "function",
                                        "function": {"name": c.function.name, "arguments": c.function.arguments}}
                                       for c in msg.tool_calls]
        messages.append(assistant)
        if not msg.tool_calls:                         # no tool requested → final answer
            return msg.content, trace
        for call in msg.tool_calls:                    # the model may request several tools at once
            name = call.function.name
            t = time.time()
            try:
                args = json.loads(call.function.arguments or "{}")
                result = registry[name](**args) if name in registry else {"error": f"unknown tool {name}"}
            except Exception as e:                     # bad arguments become an observation, not a crash
                args, result = {}, {"error": f"{type(e).__name__}: {e}"}
            trace.append({"step": step, "kind": "tool", "tool": name, "args": json.dumps(args),
                          "seconds": round(time.time() - t, 2), "error": "error" in result})
            if verbose:
                print(f"step {step}: {name}({args}) → {json.dumps(result)[:110]}…")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)[:4000]})
    return "Stopped: step limit reached before an answer.", trace

TASK = ("Our class trip is to Chennai in two days. Will it rain, and how does the weather compare with Oslo, "
        "where our partner school is? Give packing advice in three bullets.")
answer, trace = run_agent(TASK)
print("\n" + answer)

In [ ]:
import pandas as pd
agent_trace = pd.DataFrame(trace)
display(agent_trace)
llm_rows = agent_trace[agent_trace.kind == "llm"]
print(f"LLM calls: {len(llm_rows)} · tokens: {llm_rows.in_tokens.sum()} in / {llm_rows.out_tokens.sum()} out · "
      f"~${cfg.cost(llm_rows.in_tokens.sum(), llm_rows.out_tokens.sum()) or 'n/a'}")

**Notice the input tokens grow every step**: the whole conversation, including tool results, is re-sent on each call. That's the main cost driver of agents, and why later weeks care about context management.

## 5 · Agent or workflow?

If the steps are known in advance, you don't need the model to decide them. Here is the same task as a fixed **workflow**: our code fetches both forecasts, then makes one model call.

In [ ]:
def trip_workflow(city_a, city_b):
    t = time.time()
    data = {city_a: get_forecast(city_a), city_b: get_forecast(city_b)}       # deterministic steps, no LLM
    resp = litellm.completion(model=cfg.litellm, **cfg.litellm_kwargs(), messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Forecasts: {json.dumps(data)}\n\n{TASK}"}])
    u = resp.usage
    return resp.choices[0].message.content, {"llm_calls": 1, "in_tokens": u.prompt_tokens,
                                              "out_tokens": u.completion_tokens, "seconds": round(time.time() - t, 1)}

wf_answer, wf_stats = trip_workflow("Chennai", "Oslo")
print(wf_answer)
comparison = pd.DataFrame([
    {"approach": "agent (model chooses steps)", "llm_calls": len(llm_rows), "in_tokens": llm_rows.in_tokens.sum(),
     "out_tokens": llm_rows.out_tokens.sum()},
    {"approach": "workflow (code chooses steps)", **{k: wf_stats[k] for k in ("llm_calls", "in_tokens", "out_tokens")}}])
display(comparison)

> **Design rule:** use the simplest structure that works — single call → workflow → agent → multi-agent. Reach for an agent when the steps genuinely can't be known in advance (open-ended research, debugging, multi-turn tasks with branching).

## 6 · The same agent in Pydantic AI

Frameworks remove the boilerplate and add guarantees. With Pydantic AI:

- tool schemas are generated from **type hints and docstrings**,
- the final answer is a **validated Pydantic model** (invalid output is sent back to the model to fix),
- **usage limits** cap requests and tool calls,
- tools can **require human approval** before they run.

In [ ]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent, UsageLimits

class TripAdvice(BaseModel):
    destination: str
    rain_expected: bool = Field(description="True if any day has >= 50% rain chance")
    max_temp_c: float
    comparison: str = Field(description="One sentence comparing with the partner city")
    packing: list[str] = Field(min_length=3, max_length=3)

trip_agent = Agent(cfg.pydantic_ai, output_type=TripAdvice, retries=3,      # retries: re-ask if output fails validation
                   instructions="You plan school trips. Always check forecasts with the tool; never guess weather.")

@trip_agent.tool_plain
def forecast(city: str) -> dict:
    """Get a 3-day weather forecast.

    Args:
        city: City name, for example "Chennai".
    """
    return get_forecast(city)

result = await trip_agent.run(TASK, usage_limits=UsageLimits(request_limit=6, tool_calls_limit=4))
advice = result.output
print(advice.model_dump_json(indent=2))
print("\nusage:", result.usage)

`advice` is a typed object — `advice.rain_expected` is a real `bool` your application can branch on. That's the difference between a chat demo and software.

### Approval-gated tools (human in the loop)

Some actions must not happen without a person's say-so: sending messages, spending money, deleting data. Mark the tool `requires_approval=True`; the run **pauses** and returns the pending calls instead of executing them.

In [ ]:
import os
from pydantic_ai import DeferredToolRequests, DeferredToolResults

notify_agent = Agent(cfg.pydantic_ai, output_type=[str, DeferredToolRequests],
                     instructions="You help teachers message families. Use send_message to send.")
SENT = []

@notify_agent.tool_plain(requires_approval=True)
def send_message(to: str, text: str) -> str:
    """Send a message to a parent group.

    Args:
        to: Recipient group, e.g. "Grade 9 parents".
        text: The message body.
    """
    SENT.append((to, text))
    return f"sent to {to}"

run = await notify_agent.run("Tell Grade 9 parents the trip to Chennai is confirmed and students should pack a light raincoat.")
if isinstance(run.output, DeferredToolRequests):
    decisions = {}
    for call in run.output.approvals:
        print(f"PENDING → {call.tool_name}({call.args})")
        ok = os.environ.get("AGENTICED_AUTO_APPROVE") == "1" or input("Approve? [y/N] ").strip().lower() == "y"
        decisions[call.tool_call_id] = ok
    run = await notify_agent.run(message_history=run.all_messages(),
                                 deferred_tool_results=DeferredToolResults(approvals=decisions))
print("final:", run.output)
print("actually sent:", SENT)

## 7 · The same agent in the OpenAI Agents SDK

A different framework, same concepts: tools, instructions, a run loop with a turn limit. Non-OpenAI models run through LiteLLM. Tracing is disabled here because by default the SDK uploads traces to OpenAI — a data-governance decision you should make deliberately in real projects.

In [ ]:
from agents import Agent as SDKAgent, Runner, function_tool, set_tracing_disabled
set_tracing_disabled(True)

@function_tool
def forecast_tool(city: str) -> dict:
    """Get a 3-day weather forecast for a city."""
    return get_forecast(city)

sdk_agent = SDKAgent(name="Trip planner", tools=[forecast_tool], model=cfg.openai_agents(),
                     instructions="You plan school trips. Always check forecasts with the tool; never guess weather.")
sdk_result = await Runner.run(sdk_agent, TASK, max_turns=6)
print(sdk_result.final_output)

| | From scratch | Pydantic AI | OpenAI Agents SDK |
|---|---|---|---|
| Tool schemas | hand-written JSON | from type hints + docstrings | from type hints + docstrings |
| Final output | free text | validated Pydantic model, auto-retry | text or typed output |
| Limits | your loop's `max_steps` | `UsageLimits` (requests, tools, tokens, cost) | `max_turns` |
| Human approval | DIY | `requires_approval=True` + deferred results | guardrails / custom tools |
| Multi-agent | DIY | agent delegation | handoffs between agents |
| Best when | learning, total control | type-safe Python apps, any provider | OpenAI-centric stacks, handoffs |

### Exercise — add a tool to the scratch loop


Write `days_until(date_iso: str) -> dict` returning `{"days": <int>}` (days from today to the given ISO date), add its schema to `TOOLS` and the function to `REGISTRY`, then ask the agent how many days remain until the fall final exams on 2026-12-14. Run the check cell.

In [ ]:
import datetime

def days_until(date_iso: str) -> dict:
    # TODO: return {"days": number of days from today until date_iso}
    raise NotImplementedError

# TODO: append a schema for days_until to TOOLS and add it to REGISTRY

In [ ]:
def check_days_until():
    try:
        target = (datetime.date.today() + datetime.timedelta(days=10)).isoformat()
        assert days_until(target) == {"days": 10}, "days_until should return {'days': 10} for a date 10 days ahead"
        assert any(t["function"]["name"] == "days_until" for t in TOOLS), "add a schema to TOOLS"
        assert "days_until" in REGISTRY, "add the function to REGISTRY"
        print("✓ Exercise passed")
    except NotImplementedError:
        print("Not done yet — implement days_until above.")
    except AssertionError as e:
        print("Not quite:", e)
check_days_until()

<details><summary><b>Show a solution</b></summary>

```python
def days_until(date_iso: str) -> dict:
    return {"days": (datetime.date.fromisoformat(date_iso) - datetime.date.today()).days}

TOOLS.append({"type": "function", "function": {
    "name": "days_until", "description": "Number of days from today until an ISO date (YYYY-MM-DD).",
    "parameters": {"type": "object", "properties": {"date_iso": {"type": "string"}}, "required": ["date_iso"]}}})
REGISTRY["days_until"] = days_until
print(run_agent("How many days until 2026-12-14?")[0])
```

</details>

### Exercise — a token budget


Production agents need a **cost ceiling**, not just a step limit. Copy `run_agent` into `run_agent_budgeted(task, max_total_tokens=...)` that stops (returning `"Stopped: token budget exceeded."`) as soon as the cumulative prompt + completion tokens pass the budget. The check runs it with a tiny budget.

In [ ]:
def run_agent_budgeted(task, max_total_tokens=4000, **kwargs):
    # TODO: implement — start from run_agent above
    raise NotImplementedError

In [ ]:
try:
    out, _ = run_agent_budgeted(TASK, max_total_tokens=50, verbose=False)
    print("✓ Exercise passed" if "budget" in out.lower() else f"Not quite — got: {out[:80]}")
except NotImplementedError:
    print("Not done yet — implement run_agent_budgeted above.")

## Governance note — autonomy is a design decision

Every capability you hand an agent is a decision someone is accountable for. Before giving an agent a tool, answer: **Can its effect be undone? Does it affect anyone besides the user? What's the worst plausible misuse?** Irreversible or other-affecting tools get an approval gate (as with `send_message`). Under the EU AI Act, systems that interact with people must also make clear they're talking to an AI — design that disclosure in from the start.

## Summary

- An agent is a loop: model → tool requests → your code executes → results back → repeat, with limits.
- The model proposes; **your code decides what actually runs** — that's where permissions and approvals live.
- Agents cost more tokens than workflows; choose them when the steps genuinely can't be fixed in advance.
- Frameworks give you schemas, validation, limits, and approvals — but you should be able to write the loop yourself.

**Further reading:** Anthropic, [*Building effective agents*](https://www.anthropic.com/research/building-effective-agents) · Yao et al., [*ReAct*](https://arxiv.org/abs/2210.03629) · [Pydantic AI docs](https://ai.pydantic.dev/) · [OpenAI Agents SDK docs](https://openai.github.io/openai-agents-python/)